In [ ]:
#@title 🎬 **Start Video Generation Studio Backend (Wan 2.1 / Wan 2.2 / Diffusers)** { display-mode: "form" }
#@markdown ### 🌐 Connection Settings (Optional)
#@markdown * **Default (Zero Setup)**: Leave blank to use Cloudflare tunnel automatically.
#@markdown * **Ultra-Stable & Unlimited Size**: Paste your free [Ngrok Authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) (from ngrok.com).
ngrok_token = ""  #@param {type:"string"}

import os, sys, time, threading, subprocess, json, base64, zlib, shutil, socket, tempfile, urllib.request
from pathlib import Path

t_boot_start = time.time()
print("=" * 68)
print("🎬 VIDEO GENERATION STUDIO — WAN 2.1 / WAN 2.2 DIFFUSERS BACKEND")
print("=" * 68)

try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"🔥 Hardware Acceleration Active: {gpu_name} ({vram_gb:.1f} GB VRAM)")
        if vram_gb < 12.0:
            print("⚠️ Note: Recommended VRAM for 5B/14B models is 16GB+. On T4 (15GB), 1.3B and 5B with CPU offload run best.")
    else:
        print("\n" + "🚨" * 32)
        print("⚠️  WARNING: Colab is running on CPU! Video generation requires a GPU.")
        print("👉  SWITCH TO GPU (T4 / L4 / A100):")
        print("    1. In the Colab top menu, click: Runtime ➔ Change runtime type")
        print("    2. Under 'Hardware accelerator', select: T4 GPU (or A100 if subscribed)")
        print("    3. Click 'Save', then run this cell again!")
        print("🚨" * 32 + "\n")
except Exception:
    pass

# ─── 0. PARALLEL BACKGROUND CLOUDFLARED DOWNLOAD ───
def _predownload_cloudflared():
    bin_path = os.path.join(tempfile.gettempdir(), "cloudflared.exe" if sys.platform == "win32" else "cloudflared")
    if not os.path.exists(bin_path):
        cf_url = (
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
            if "linux" in sys.platform
            else (
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-windows-amd64.exe"
                if "win" in sys.platform
                else "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-darwin-amd64"
            )
        )
        try:
            urllib.request.urlretrieve(cf_url, bin_path)
            if "win" not in sys.platform:
                os.chmod(bin_path, 0o755)
        except Exception:
            pass

threading.Thread(target=_predownload_cloudflared, daemon=True).start()

# ─── 1. FAST DEPENDENCY CHECK & INSTALLATION ───
print("\n[1/3] ⚡ Verifying video generation dependencies...")
deps = [
    ("fastapi", "fastapi>=0.110.0"),
    ("uvicorn", "uvicorn[standard]>=0.28.0"),
    ("multipart", "python-multipart>=0.0.9"),
    ("diffusers", "diffusers>=0.33.0"),
    ("accelerate", "accelerate>=0.30.0"),
    ("transformers", "transformers>=4.40.0"),
    ("safetensors", "safetensors"),
    ("sentencepiece", "sentencepiece"),
    ("ftfy", "ftfy"),
    ("imageio", "imageio[ffmpeg]"),
    ("qrcode", "qrcode[pil]"),
    ("static_ffmpeg", "static-ffmpeg"),
    ("einops", "einops"),
]

missing = []
for mod, pkg in deps:
    try:
        __import__(mod)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"📦 Installing {len(missing)} requirements: {', '.join(missing[:4])} and others...")
    try:
        if shutil.which("uv") is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "uv"], timeout=45)
        if shutil.which("uv") is not None:
            subprocess.run(["uv", "pip", "install", "--system", "-q"] + missing, timeout=180)
        else:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, timeout=180)
    except Exception as e:
        print(f"Notice: {e}, falling back to standard pip...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=False)

try:
    import static_ffmpeg
    static_ffmpeg.add_paths()
except Exception:
    pass

# ─── 2. EXTRACT SOURCE FILES ───
print("[2/3] 📂 Initializing Video Generation Studio backend engine...")
PYC_PACK = json.loads("{\"videogen_server.py\": \"eNrVPGtz27qx3/UrUHamIXtl+tGkc4466lSxdXJ86ldtOWkn4+FQImTzmK8SpB/X1//97i4AEqAox3adTpsPMglg38DuYgHGcZzBT6GoJif77IyXN7xky7xkn+OI5+wTz3gZVnGesbOqjuKcuXs8ihdhxSO2myfhnH0MF9c8izx/cFLmNwAlGLwWeZxVgjBV/K7aqPIN7MuHLE7DS26805/2fZDmRG6RF/dDFmYRW1yFZbiogC9+B8/ZJWdFfstL4GB+z76EGdvxtzd3/B0avRcvl7XgpfAHDkg2WJZ5yoJgWVd1yYMAyBd5WcHQLK9IMDEY6DZxny3iXL9epeFCP/8q8kw/50I//bPmNdcvgi9KXjV94qqu4kS/VTwtlnHSDK6uSh5GcXbZNMRp01nXcST5LsLqKonnmukTeJUd1X0BwLp9koGq9uJFNWQHsYDf4wIlC5NGtCovFwp0CbYOi1jDugMG//Z4AUYTQ3pRs0G9ANfqKS9T+fQz8M5L9TybnUzvFpwoyqZTDooBNujlvEjyMGqxfOHzsxxmTNV53YvFIs8yDkIMPItTP42jKOG3Ycn9RV4Kzfnu8enZYdNlg5RcFGBa3gxGBk5V45D9cnZ8pN/UFEnziCfCn4eCa5jZ9PRw/2hyEJzNJrPp2ZDNQnH9Sz4fsl/zeZCGGUzk0oJOYW2EARq+Ibx/OPk0DaZ/nwH85/296TE9W1A08S951jHJ9KfJ+cEsOJp+msz2P0+Dk9Pjw5OZVNvnycH+XjA5O5nuzoJT6D8+Mzv2zqnt6Cw4m+6qDqT9aXoUHB7vTc962g5UY8SXYZ1UATEXwAKmJ9l3yatAcxsswiKcx0lcxVxNnKLMF1yIdgjoCa05mBwcHH+Z7qHo06MzZIyNYcFUbqsdj/0ftbQ68gaHk78HfzufngPkL8cfEQacipsLn2c3cZlnPvDjOq0U9nBnyJw/Op5Es3t8tHt+ejo9mgXnJwfHk71nYVuFQqQ7hBRss38yOZ0Fx5+npz9PJ3vBx3/ALAG021vs9/Cz8179GQQfJ7t/nR7tBftHMJWOdqfB/h6Mw3Xu48971/Ov+N3X0fbOxWDwWzapqys2y8GtokrqYhBU+BIksFIArnEe/gE0uN4ArHyGWg1mx0BmxERVwrCuZOcnZ7uTg+lpMDmf/SyHOh4DB608ly+J1GUiwiV3d96D4QYwG8jqIbAkuXA9tvFnJDEim9/GwKvBn2zFf4CzLjNmM6dQChsl/RLfhPwoz7jEc5nk8zDp4sCeeMnAgzOCRCGaFx+wxIXrGYyEMSzqz2FS82lZ5qXrkIIlaFqLioDnnIGTru59x/uGYDY3aBCTrpIwgEAaL+8Vghsk7i4gQMURxM5R46G/AtQFCT3P82RkitaO7qr0pzABv4UN/K4AjwmRcLxipYEBgKEMfGdagKMMovgSvHPLjGZ82GBrZIDIjaE3mHMALF1En5fx/1Lc7JPBarGEsUG7AqHBpQcJS8gaxvZwzaAvCvA2rsNgEW57GnsCwhKYx8ZjJpMAev+6deEnmCi41ONIIZwV4nLw9kVjwC5jgwFlBugZof2fdQxKRAZdNZKi3cgOe0+pCuSTEdRVvnaMZDwJeBfUhViECS8DtSqeB9tZOI11AeBJM5IH6BI1LffkRF5ZZVY64EIwrmoRLCCAjN9vbQ9BhVUYJ2PnPNNM8GgEfhjQxhFyksZCYGYDoVtgGiiXluOZqvYRLfcxywPpHpxC5p1gWuYoOOdRT+E0ROEwBwnm9xUX0n2B55ecp/M1caDxlhgHpPMPDj+i+/+wtYUBwJglQMMF4dK513H86Mw33u4fYPuSl9cg9u/YJCJVgYr+Rlno2xIaBJdFHVB+O5Jprk9kQFnGmwuCCzDhuBN5ISLdEp+BDFXGHG5j14yecELTItMQ62KcMiey1YzMGzd/ewX5HZuVteEsIfkY6ZQNV0EjERnYa8ZV5X0LZEx8gPdjmLxhtuBJAvPUGrQu33F/VQmiD34lTEVLh9OyYM3qgP0Gtr2SNo5Iw/I6WMKC4pG7dNTUqDPYI0FSHDGOwW7EHoDIo9PysYzBEElHZkM7FagsiMAmjdZ5JnDzpPQOq6+EIKFVr0O0ZXEjghp2bUmCmDYAiwXNAx3K7V7UBXiIG26GdWLbRmLOGzm/XGAWzD3uThxwRSFP82yMc2bIsjDlY4f2vLDl/XRyLpVpaG2Vmk+KICW99SoHBtiMJzzlMDnfemnrlA41Umki0i3iHhLDzBC3lG0Ap92jv6ijkAxxAzMunCeWMVYWUcRv4gUP4ugOrLLV7QpQ4ZQ5NZiRJQWEnW6LwDbCsuQ8SHk6BOAqTPDRRgQNASKLs2W+Fovy3A8r68ppxINwQpNjdYhECv1akp4xN7D4A8XhHIaWeZ3BbGxY3mSuESk8zGjWYZESt0i0Br6N43Gt87GNVYRCDHrVYqmD8s6WQqsHZxcmLPQm83Bx7RgjulrY8re6va10Tefjd1hSusQ1KQq2x8EHxqiGt15bIWAfa1oyP6ziKtG+paeepspnSmmQa2FMHzvbPmhDNUb5QuC2jFK9obITNBptHpH2wygK2mqJpG+XSVR2mkBiHED6dRlnYvzV+b1zYXbAdjDiWRWDucftEpB94C2u8qgH6IrS0qYD3eJfkKc8C/gNYHMd8pd1AU61zaehVzVrb7Im2Gh0lJxtWjjKPFerXaFYncWQKuqp+jxDyPkprYFQ2401qIdnoDlCp+qPsBbl0w48NTVIE4T2Zmr7EFm+5dGWLSzizZvtTdBnUl05mDNjbY5ni5iDdlWlzjU3It6FqQ4JGCyuOCZOKkgXtdoidrz+On1R4o7y5dfP08hcqg/sALAZuVwY1Vf5MICAGRgEv6aeFhUEekylkL5RafNVT17KRCWCfUUN88rweQ51UGVqEVf3AN/JTJ9SuFnPeoXaTfDnqZ3cH21AoFfyYvvLIbpEy0RrC3CuQvY9shHMn8/qud5vTFV1/3vkJbQPhM1dK6TMot0yvFWPo06WMtR6HLElbPOqtYkMljGx7FiVBjapduzCrV21cwNbO101sAoCkAYRAkxO46xTTn3BLnhnp9kFL2EbLOoCa76QrRP2dw/45/GdzyboVHFz/JDEQpVFG3r+Nb+HSeZhWr9atAUh+4u4LlVypb4u52M9/zYpfdAzTaNYp6eE1oaB3utVVNKrqYM3U1WidJWAsrQWpLSwxELgv0P3K3Ve9NmyBftKEgrn4rVcHtqcQejmgsgo5jtW1oyHAgtvAQWkNao3h+BU3f7j6EdnVfUWJm2B1cOCt7CCRerdg/lqiGZtDKJaxtxA8IWqu3TlNIeAnB9UpUXlr+7svpBF3KFR0PXWUvig9WK1W3qxzkreQi+aFOjEpPr4TvxJpkq0qnuIP/bqLKtTTG54yTGqiooXWCClktPWkKVx5n6Av32q7IEEje784HkvVGo/Czs/9HB7WYP/plFYTtSM+orT7Q/4RG56hVsbEk0PHumlnK6QByQ9XApOJfM+pWEXEN/YfiFlhXJje7AmpaIYM2LtYVrTmqjmxGgvyjwtKmfU6wtUJ3gBwwMYwBm/DClVehJLd9Rw3ZFjLw3LH40sZ2CMKrnIk7qSKWMfE8YAEOf9D1uwPTDJWO5gZC1jU+CemT7qm7Zm7mnPt1Fn7gzNzQMl7PjHaIUEP4AcETJtDQhj8BBnRcaekUO5l7ZELUAXeP8CcsuIdNqLrDNqSPsIC0+K9TvK0II7rBl0Y7jZvQbu/mm4ew33aJ2NBCIEvlSxHW87uPgzMq8AwDhRBVFcjugyA+Vq+KD2bUCS31WwjLCNoH38oWJRXoLOCZOfFu8dmJP1chnf6WSNEODBpcKgcYHvbx4z1nMQzcEOzCGcKpUCDvHiBaVRkltIkZbOQ/e89vFB03uUoHjWQIcMWG5ePXiQRdEyriqeUVFs1TFRyTSHrYbbsAHL4nbueFgtzusqWNqVm77Ct/63uKozrKWHt2FcMdImlUTNmtEKkD58RNhVlPhvDkiuV3q0XP8zphM5gu/Frgf+uVVXP52+CNw7kCxvhubtPwzXDmxDNtmG/DuPBDITp3WqIzTDcw2WL9lDa9TNTr3tkR1+9J1+Uquik/F8lJ6b2lmpxhvWWiS5oCp8R3VjmDxWLb2ZLD6/g92C6JbJ2/46S+Ls2jj9+Pax3ZZxbEc6A+0ge1iyt07MVcxriOlddpELY5utd3abAreU1St22hLQPnVRukzMo/XW7+ApE/51jYPWkuvIQDfSng0nfWKAN8JWz2bxjpQ5eJHEHBymaHbPQRw9DeU1VXfjSGZZJ4n7krPWnR9bo6lLffNa3I/aa310xkDY0Y5IwAc3cs/CyxD8pLiCXDa5b7L49dtvPIF91CwbutHn8Pp99VrGypFBSwSQEgzaQbgGlpXTNBqG17n2OEpOWVnvwdpL5vnS2Vcn0pax8SDNT8Vlk6dbRZ2nCj2qqhG14j2j0DG08a8vEHlmeaNLS25pnQvJMVZVA7rjpmLs2mPvL8enfw329k+Bhr68iCPwGQKiqxNzFeEwRo4N7JsYq9WNAjlEOJ3xfnqNiPBKSlapOjMj9xXk1/SqHV9W1NKhrK6c5uIITD7ySTjrrLzByNAhO4l0ZJdetj9jGRpcthOuZUPt0Ft8rYe2yzRUc7j46ihXJiDTBl9T3hMp54KYlXvRRsLB6+broVlW0JdUBAuZoihJMLqfKPXjtEx3PCGx1WnrUSeM+JYyO0j69bo6YTtgRMW50HURRfbZOu9gM9Tepax3JD3Uvbc2TKZ0wc37zQ1peUm6sdEi4WGGQQSiB6jB7Ysq2r+iRvr65VXqpwApEaYopDVr0UV4sxofCxNPc1PBNYE86zZGA7rIM1GnvAUyMNnwZpJjLEDkBTyXmfG0vZ3cZyXMUDImdCZkwFmjVHw5PqOQ0nP3Qx+Zmrt9B6Wkgw86ljPPURpZcZ+bciFwLuLJqW7Hy+zSPmzOl3nJKTrjTQa8zKRUiIHRVCSk4hVvEyCSF/dp1X3Bx46OLUaO2oo7NvSJeytjkFwO4+76aAf0zaOxaTg5VE3fdSd6K9YxMh5kLcvRr+CFGqsUIwf8BBmLNbus2zB20iPTHAGmKFEc3/G+cybVpE0r8wIZlS/m7MBW+fL4nc5vdvOsKvOkObwRb316s3qehid4mw9S2MdNJevLs31MqhCJROBKfPKu8GjNslAgaqw3sC9WvcSVv2+Nj1pEDEu8g2HveFrzQWhbVO1htb3zsRVShLXgr9AHwRni/YeoIr8eskVdluQXCFBy4hO/rkU5v34J4R+NcLpL3+1IHZC0MZbZbboQZ+UFUdtID+1y64x/jrUgaEPEeoW5JOB/kb0kw29sMIn032oxGXBfdZCPgP9FFpMMv7HFJNLXW0wnPJDAOJRL2pyrI7jOJ1YyB32+sSWNDZmObLTpyOutvpLZuP0lnHZG6IJpX86tLrM+lXa/bmPh9JKjNI3jTgISEX3VyDemjzlt2iTyJZJrtjtm7xuKy/YJDGb+07H0E2lElN9muG96hYU1aIDOKKn+sxY3IG2zCPabMayfPC0SMGbk6GmEA/K6MrN2Wur2Rqgzynsdk0vnKGcNC0wiZM1lUPqwF9X1TlumPfFvTmxUianLkI+d5vQxv810jU0I7FE6sO0ORH5paWxzNvEAZ2hccpdMjPVDsyN5+08y9Ces7JSHycYsBtFPyvwSpplgv2OzqzqdZ6A3dlbBdi3Fj36/R+59y+eC2FifgSN5sNLzht8K65IlNrcwzdPI+KCXra4orGbK7+zGrIHxYbNU3ltFFxrjWOurATWORvAKpbxnauFTV0/VRRnz6yZ5Rj4wK8yanb6vohTyb/HRL5XFxd2Grn9uaNm+9VFVg9XrHga1NOSJkFyt77e2vU4FYvAvurEuuXCBfsI4LeoOEHjlE6vj7oODixEDP312Ylc5luTzWl/Ruj/fefS+IW2/kE+yil94YF1SftBP4pd1lsHSU18NyXoCXkuW9QaVSUGgEosynnMXh/XdxqF742GiMy+dqWRhIa7ySt10fo6+FKJOhYva1JJAdXqYJLlNLECtNuptK0qdipeppqfOiPlN1RRvDV10Po96WgzA4XU/YIK2f1UE+5xZ38Hp+88CGiM3hbTuzZymUrdy1IomrLPW6IYSnvg87Fuz9BmfWvw/nvLtzg==\", \"models/base.py\": \"eNrtHF1v3Ljx3b+C3T5UAjZynLsGBwN7QHrNFT3kklzi4h4MQ9BquTYbrSTrw44v8H/vDEnxQyIl7cZu0iL74EjicDjfHJLDsF1ZVA1J6rs8ZcURE68N29HuuW3Zpnu+bmmrGpqriiYbll8ebatiR8qkucrYmsjWt/AqGpq7EoC67y/yuyX5O0ubJXnFavj7pmxYkSfZkpy1ZUaPjo7OXr779Z+vX7yK35+9OHv5nqwI4PmD5jVtgk+LtNgBWEM3iyVZbBOWiac0yVOa4ct9CDg2dEtimnOC4zrZ0uD6tOMy+g2/Lgm9oXlzyok5r5tqibRdhOTJj+R1kdPTIwK/xWLxU4JoSZEDx5TU7bpOK7am1V9qgYFkRVFKaZAblvD3KIVecV0UeSxakIYIsHGsbEuuo22bZUEohsFfU93pF/xdR5e0ifPiNmFNEKom+jGlZWMz83JXNr3eZVLXRwO811HZKpycfIHYhfRnIFD35PiOjtIM/iVnSf3hl2KtZHTGeXyCTJJ/vP0XKasipXWNev93sSa3rLkiIGKQNPyTNBSISvKaoeZrkuQbklZFXT/hkhRC3dCM3dDqjsuMD8NVGrOcNXEcKLJqmm2X6g0Gi9nmFMfSHxsgNgYrpL3vLEdZoN32GsqkSnZ13zJ0e5oxIDEGU9gxYBJ0jGN2how9LsBo0YhEH0PJSG4kqAQQ8WA3KmqhXT3bIJpwgNEvNpBgAgDEg93o4gBAXZ/tjkXbaKG5OLbBq+QWNNbQakc3DPQ+uyNaSVtzvUD7gvvxZtHjsCouK7CyU7LNiqQBuKfR0x6fbVUhR1sQAagfSEGonsCLJskERO0B2ZZjg9AmiWuaFvlmDKqs6A2jt/H6+fdzBECrqqhiIKlOLumcDqwG4bY1BUtcF0UGMD8nWT0EUoFyDC4Fd4YQGyeN5ggnhQj/GLFIAHcRmYMrSnk/RavdJc6K9APi7OaQ6N0r+NLHLDiK04wmVSzigtnnJX6Z7hPhxBES8mcAQASv35wRIapeTx3bQZE4O53zKem8i4ov1iD9JG34uK8gVi3tgHmB3J5f6HClEAY8TPGZQU9DA3R87rEwGoG7N33BSNZ7sEs+1uwPujp59oOWiAi8SuL2DDFgOkrKkuabIMg4b9ehRlTRpq1ycq15a/M+d30Se1Pp/vSgNIMMKIFv8XVItkVF1Ct4qqMH23atNckLyFgMbcR0B5OeoHXO3D9NsEVqBgYTDCgyrBPCDJV2ZZuNsJoOjnMpFMB5NIzSGh1Y5ZkG+nRWgDWbyYQ1aKfX63DQDlGrYXlLrYZBJoI/b1YTWHkWUC2law8mE4x3bY4h5CVGt/nUola59Kweo7rxGtQACn+HWhmaGEIgcQPEhul181CdJ2V9VTQiMHFr4UZn26HmBJKfd9zzap58SjQyh0ogdSK/0/V7YJ42TzAMJw1bZ1SmUMhITm+zO9RyTlOI0McVVc9ytq9VVjopU2RbT85kBROzzsaH4pcx45NT4gtMaxanGsNi6YYTg5mQGx9olxAA8MlTmIA9YFZSALDDTMHT0cwUun7mN083yB46aHj0ABlJBAD7iTfSiA6p8cnTaVPc5jAbb+K2yqDXdnGclOz45uQYEtD6+JORlN4fd7AOId9PWYNcjh1qCjznmbaDbtE3aQRWlviVGIPM6LoeVppHwGEXsKwigsPoEBXodfDBDqkwTHvkFOj/gzIUk2R9RyBrrKb14hW0ErKUYV+DMJcE3WJnCeLjWeoiJBQydEOaw/G1Ugx0DjBDIVXRwixrq4U8Cx2dDlHLASqRgdKgC8OlmyQ7XIoukAEHGDgHq7JwSU5CpzD2i6WGvSzeCtW4XFBqTSgtWPyesIZvgOV8a4RrN4oiZ1epetEVM9Ee2ffG2oL3ijeQOUznEaPTujk74GbbrNlhPBnQDMz0C4G/04J4cxlVtyGiLKr78M0lHsQlepmC+eoa0gyYzig6Zr5VE+t9Qm3DuCtxul9Cam5rYNzuhdTehrJ3YrQ3QnoDCK/uYzf83bP+6AAVnBtMj7AiZ1VLPYuZ4eYGfw56iy3gY5ocJfsZJA1FM7nhYlN0gzsvlpf01kL9bRy+WIdugz0ILh1lR3xwbTxiwwZNaIl7hxenB61shjaj9G7toI0a0hJiv06Q3LbKw6QxcjiV3nltzRDO0m1qPWxy6WqlG9oiwineDFRH+xnzDK/Yy9a5ZU2nWmMTREfEt9nh8RImTKWdYr7fx/EN21YhABTR7h4uBjzkZPI4MUC76WQc8MT4HtY/zYgs+/r9cMaYOf/sO6l8rutrMr65/+O6v3DUz/Z/w270Bit3pzkxANGDRZknVfaqaXC89nmZw6EO5YoVU+6uQ43TN3XM8U3LMxx4lmfO29aYTFE9/LgPSbHDT8O9GndP88zSc7bZNxhnqPlv7JzNDjxfeA/NJf2xnTIwEiVbeVxnH771Y4IE9gQG05FUXMA6l5htpVX7FpnmCcjf0MdrQrdbluL5SHZHbq9YRnkhC5ApMEWkO5fBwZEVPJ9BkKukJmtKc71vGDnHEUix956nWnPylKmd4P5yyjEGqqNXWDCGZn604KVH6GtF26yeRs/+GhqHyuWGV4lIM/eV+fQqP4zKnEG9h1EG1K/z0E26vkN/g/lw8G2yhkPW++x5jnzwHDKUdycZoGfH8kAcgxE103N+gg4oDMMhBkuEiAb7LlFegdXk6muK2O5qtrh6gga6DppO+OiCNTKVYR9oDAdzkaE3f6RxVepg+ZR+6yWfohxlJP38tl8/mn/2lTkr67RU6dWssajyZqievJNXLqiAtEuqD7FKFmTxiFUEB95/qL9/7koT8Bik8DrKoo7wOaIfWd3UgdEcusO3s9aj+wG2Ns9Y/sFC5ASX9R1v3ntKOwZlqXOimcndyuR1bF2pTyFGwyMPjVPBxQkyM3ecESBGah9m1j3MqHn4+sOBm+y9V5sPUOWwR0zoHFfGBFWpKStWHbnwF0wLJpdUD7pE7C/L5NOjetLIemrmsuvrnmv1MstzsjbXcEVtSVcGKTRVXz7YPObJqXo+E86x3Eexf1k8NG20SjSPaba+iqc51U7/K/aqBLmHxcprJr8U61+THNBUzpsmFb2ELKe645WOGOSFua5RGBCfwBgAG6zReGaUVA3bJmlDcC2at6XnYonwC9V1WNj/3XOcZh1XOqwLI/KCDK4LP90fVPs+IAG6DL71MA/vbzRF3N2J0cTJBeuAMOWfBgaLK1XS/8k4yRd3BXAY33L9y1/A4VvC5p0lIyNscxrTjyWrrLik7ujg/bcI/3wfhNEV/Xh+evL8wgTDmVKgDiwDFxhWrloWJZCVp35FS2alH5e9+10oo5X4x25yiWfl+qi77XNlAG39XLB1IS4x9cO2Uz+e6oERmz134bkY3puSW1BIiLJKvDqHJin82bgXxm1BmUrnpzNnPDmSEgPe0AsE8rB/GDHkTBLjtt6OtANPL5W9TkqV0+yiYbBtInFi9JSPzOB9qFDhDAMbGcAVH5ZEzcgAFMnTG19WYHRw7DA6o5bXdHyTtfcgRWkVshwDWz//nxGSHBvPcp912l+cu6xzTNV5009danDKzjBkeGt31An2lfM9wl9UFqXTAZai5NLcSNKygNSB3dAYUhlxkBWnkGYYpZcsb/YMIu0uOBGpi+VX0U2StbQOQu6CvXKdXgZsBB17InOTlRe3HheQ/bprbzp7MS4mjTLF+UAR8mOS3LgNxXliDd0BS6FzEwtDgL/ySoYfO+se3cVVFnZLngy7/ujJr/x7VlI43cUkzRUaEjAdht6ukrt5E+Jek6MY3INbWvJAQ3hrCVub2taRx1FGtSbEC5j2E+iUY3Yk2gzIDB+p7kx1wFwZ494LnjWjWIy8iavA2D0UH5x3k92scswH7J+W4ED4HyEEHMGolZTdlm04bhdltyP7gNuwMiBlNA+kcHEJhma2E6svYEMvxWDk/wAu3rx/\", \"models/videogen.py\": \"eNrdPe1y20hy//kUE1zlDHpBSKQonZcVbkUry7Zy/lBJsvcSlQoFgUNyIhDAAiAl3sZVeYi8S/7nUfIk6e4ZADMASFG79norqrJEDnp6eqZ7+mt6YMuyOp/EhMfsNY946ucijthlvpyImN36wR2PJoxHMxFxNo1T9jHJAj/kqQJxO53LZZLEaZ6xqVhxFic86mXxMg04WxHaWYV2EU/8UOSCZ6MOY32X5YMVq/3873/+F7viD3kvj3uSLvsnP2IDtw8dDn50mPw2YFdng0+9Q70Bvh/3h9Dy9upvvYF76LA3y2i99COJp+8edmHYgctE+7BnC3/GG+O2jHRWjlQfoBwbRzpw2ap9JIJvmeGn45PTHk6T7THCHye4cH7IXvIkn7PjaJ3PRTRjnwYsiKOJwKfwHUcburC++N0L4mStj/aOmtkJNu+xq9SPsilwsJzfcSQWfs57gx6MiagOXRbM/dQPcp56/AE+RzNOqE6KZnbBk9AP+IJHOSC9vPeTlmlUS1Ybo/Mx4xnzWQbEhzwH6j5dHL9jCz8CFqQsi1kchWv4xdlETKfLDCeQiISHKIhxECwTkCL2+vwjW/BFnK6ZnwO6XCy427FApDvTNF4wz5su82XKPY+JBUop86Mozkkas05Htc2C4pOIsoQHefE1zopPOV8kUxHy8vs85f4EiC8bYGQ5ZuLn81DcFgOew1f5IF8nyDrVDqx02EsR5CAyIoPfHxSrHXa1TGCkAnOwGhQfo+UCWOhnLErKceM0UPjPz94WyEmS1RrApuNh5t76GS8eX/nZ3b/Et8bzBZ8I31vmIswKMB4F8MxLUr4S/N67PRo6sMmBRPF37tHmBuFASAcYd8c9vuKRNxELWNk/sd6X+wFsJ37uh/GM/Zl9guEnvhLoKMv9KM++8HCdT2cvTz+8Pn3vvfvw8vRyRGy6zvLU0T4B/25u2Jj90sFtZoEms0bqi2wQecihyUJlBmySe95yKoiU/7wUKc9ggWFvpGsPBQx6vPLDjGtw+mPvTkQTgHkPO6MNVcphZwPfuCdQBFqwaaPGiyQHiKt0qQA+yz+W2DgZEqwnzabCvnEylqT1S81HgygmtNo4IamAv/SEVhuRfRkGaYp+w7w0nf8tJlXDtxOTmiZnw9wqI3RaAP7xp/i5plbe7qZX7v1o0PdAu3h97+DWXJDIX9B6FGYXHCDykZj9ys9y9j//za6GYKf9ybprzimJPTFRPXvHZ3vwB/r30IHC/r2XZHJ5mund0E5k0OmadN2N/kRE3ir1F94MCXzhDozBgngBTsKETzSY/sDdd3TO8hTJyYcemlVDYkMR8CijeR4nfjDnMNV9HSACY7Dino/uaMt2mvCpvwxzb5og8f0jXQSUP1GyHxYbl9oy5RJZMPByUIre4VYOSE+RHQIDfuTAgOMw7F3ES5h7ugsDBj3laO6y/o7U0pvZ0N83lridD4PhN+LDYPgoH8QmRuBeGD7GCOADeuiwFUJ/ls1FAl7wKW6JwfD1j9/tyA7l5v+2/TAY7MCI4YtWRoAu8QZDgPiWjNjAB7ELH86afMCmp/HhbEc+iP/XfGjuh7kMPpUr3vcO23lRj1GZ/QKNxAkMACGZCDYxIUdzF+V7en+wD4e/RSkNv5pSUlSCx7NYLCORr78SK9Si11gR5g+DQ89ftXNA5QSUXf6OvSdy2DElWR7bC2/FbJ6nIrjL9hSe38KAox0YcDD4NZuAiNtp+Wue01NWH9a5RRn1vZUPntnjHhImJpSL9AnU0J9l3uLJjlKZ3+hNtmukleRJi2e7mUXf78ChGhv/SP4TcmILi4Y7cQgNxnkKQZnBpJ1NRsGh4Vdh0B/bkPxqDg08X+boHrfqZTJPcurVEhzdH+PJusg0/gqWDXpGhrCdW3rY+1Su7aT5NpieP4L5V8yxjIDy+O3ZS+/48vz05Mq7OL46+2AElZRHvBZR7jD4pUeV/aPR94DYfnEwcEBU97tKDr4f9Y+wHZocBg+L9v6oj81HQ2iGX0XzcHSAzX8ZQPPhcNjViHr5keh5f+ldnp7AwPYBgADObufl6avjj2+vvPenrwHi06l3fvHh3fkVwkist+EyTdcOm4gsj9OcTxwWxvfs5yWeHUD7FJb+jqcimgEMn8bpgk8YsH+SOYw/5KnPQrG4hS/3sF4pBPl3DssxCeenuZiCwGRWp9vpdKAvK7hASVAPcNEnG3+NGC2iEo4RDBv7OVDZP3T3u6z3Az4eSYot65KHPMC0MGdzsNcQ/fUUvQzkLAFRuA15MZpMudJxik+f2QwkJWL+yhehj4CYDHcxjY3oE8o9pCCocQqhZMlFLe94XQtSnUbiwKmHT/pOEZuQND1Mp+7+63hWGh5N4TtNK33DxLRYWvbDmHQq47BDar0J1tAFeuLrukVx6dCtGaW2TjtQiPrjEQo/0+8AJBEz1DwDTtV45854TsLlNFb7pku9USju+Bo2rIZoVE5pwXMf0NYySYQWenVLOKCdQAEFs4tZ/NOY7TPAr80Kga4NTXnTrUbDn5TnyzRCmjra14q2617/Rm0moEJKyowDi/zEvxXytE9SkMe5H3qLYivRHjLzXuV2uihGqXAQqTCiXxzhwKa/F/kcNju40ngQRCdIxW7DXutyCxUzHrMU8zEmPXjWtk+an/W7JdfLpz8wxfd9d5+QyQMTL17moxr9uDU/62x0JBeAmXWOiZwvMltba01NjCuWjcc4bBe5Zm9nG+sB5GElARWV10DIjaE1dKMuUdEXbevUTLUEKr7X4QoDraiibw0Ywxi30N8Yu80+F3Q0n9X7K9MtO9CXOkRlwCVQ8b0OV7PjxXppjfUeFSsBvvpSQX3Wt1Jbhvv4jG04k9c9DB6hsZg0I6qCISFYUNs8zHJBGEDuujXokPhXioz21KcTUY+IqHA2HY8WxJOlJD3zMh6YXQ33wOjDcxjOYLr6VOhYqWtwWZegvavqAjx5BAbieLZOtLLjKc/icEmAoJQznlM7bAsLnJ3EIm1Uc5hKdXQiB2N4wNlbgAUXANeLp73+EatGpU1fnVarc9VCBeH5q3fvyL9z1OEta4h6XKfd0Vw0ubWp+gK9EPcQVVVjVqgxLHDIEksqrb5SWvfQZ+E/2IPDI5qcLRWhLclizyXiLupCMHTdLrT0j+SQ821d51u7KhE3T4dtWIU5OmBf+pD40igm6F36U87Oi5qBd7K04EsfFQehn2Vys8JefYc8VyOV0vMmDicZ1ics4iynkgbwSdBoyfqYlvoGocoh0HrzFZiYjN2io8tZBiYvwAoQkisaAXeD54lI5J5nZzyckijjAXFlXLDZ9cI4uANelrUL7ltosLs1KEmcB9t5VNYkoIlDE4JY28EL2kdoCAvIkj6axCbiyIpXFJruB8h46zhMZAzcgBqq4idP181GhW7uZ36ep3YrWnAE89jqtnfeOGk3j20rSJZWt9EPHE+e5HhQyWkt2zEnIESdnYaqM2ET79rgZoEbxCEGKna3vshUR+IGy4nviswrIxG7ZSk0UL5I8jW4ehD+2s25a4td9cEUZRJ4ipBNS62TU0HDGKVIgQs5KVeFmOlIlSsll3T+v8fgbKpiFxI8kM1fKXe0pONqBNqbv0IylUZs7dkxt5bcNN1dfP+SLCMCQCqwU82h9wXYhU9+uOSnaRqn9tT6GN1F8X3UWrDHwxF79ks5wOdnVreiU3mEbFz3EEsITGGUj83ERgVEAgS2Ac1xnLk8Wok0jmhm1sfzy5Pjt6cX3ptX3snxyZtTi7zhSr/gDzDaXSYYj+Dx+wwsYWYXH8aH6NkHS4jCotybgjPBx9Ag/Xv6mo37+9AyTbIxHpoyoBU/aPNs6BNVIVVmFxn7Eyz3z/6IvRru9zu13X9G0LTaWL0FrW0suVhGWEUmmdIQHFWpojGnqDhgz0oyfgC6Dw7c/Wckns/8IOAhgvNnLrOaKM+wfioMMW2xAOYD9cAhLIKj1t5HbX4VKlgMLCDEtAu0Z2Bkc3CAYvwMC4xnRongAWfTfAphLBZLiPh6Ol0kfHZj0tBlVIAGy1Gt9CRfJ6jkpAK4pUgRvCyQZnujiqLJmk9vp/0jL5OVqXwCbqn0hiSQQlqNiTuNe1WQWER6T9ONXiFUC1jNsd4LGjwMjkU0je19U1E2Bq9w7DEbA1P2/Dk70IQRdxHASRUiZAmcQNEvuVXpRdxoTrFNHbm6TrXfNKy7z1NM60T/07bAjLhT2AEkzCkiFwgLfl5yJD70wHp68RQ5M2mzCdjP3dqrZn9g+tMNo0pd9oQBGx0aY2W8HQNmgBAR+QewrlbD7tZIXPnc6jZWzIXmiv5cAGtnG4kGWNeAtDUub7V5LT5Eqfc7u3gm+LFTs3TURo3/nGHhbQCiMo8nleu6gwSXIRyJsfpGsjxSQlsX7Jrr2jT+dQXe0RMxCwin7u79dJa1ZXhqiQYc36PxIVaVdNQSDfE9CQ9qgWXWVjz2Wd+F1RwMNAZZ11YJZSFN5TdjP0vbO67KOGoYQ8xQgl4iOStXwqGDmSJuspxqjdyXRahSPO22pQoBr4uKHSNSsBUANrFLDfT8uTGRboNeiH3sqt5BqwXqPoF6KlO9islkajP5bXP9CpMtmENHck+YHx5vfjMOFUQXJ1JPoFud8X1T0osCjl3J1stfvinhWPuwK9Fvr/72u9Fa1/hbRnsK1k55lFDcChm35lnsr5FKesPDBGYA7jwGCBC+53iQBDNx2DuIJ/Eex5/VnZwT7R7Ol84ukYHEbI06gZPhii2/4EUTZQsX/oN6NmKUvsxhEcHhzMTfwUTW0ppaqhNvnlyTsnTp940jz2ZU6jPwEzQuq4FLC3/iJ3iZRhteSo8KNQEa/MYPCUfGatq6JbqZWifxMpxQN7wxBnQly1zGnyP2SzXAZ+UxZWmAJ+Vk6hIKDJGqk+NzPD0+916dX1JUiKc48vxFLUZjgoDhWgaemIVEd7taqU4j1Lufi5CzkEe2RNhFV7daa2PvxHcOu52likRkWjPLQvO9azputwB+Z+5EigVwksEqP4lDWDRA7lDLyYe3Hy68H19fDC5e/2iOkYgQuskJ417z09Rf24CsCxThHG2ZfXUUzAXP/EUSUjLw+P3Jv324NPHJibp+AkyagC8aqlNKvAUUagslpxxyPysSQWq29ZVqEQYIQ0vm4622nFQD+ztPY4bLSCfiihDLyCvLRiWztpKR4mjfu12KcOJNRZqpsB8USXZnqwmUrSOmyYf0yRBObS/DjZRPkxh2kfegPZN7Rn+6bnlK+04bq0wQv4WQAnx+BlsSPsHUcXwU52wdQWiObGM+KGUOMK9T//YEVivlM0rQIKRPCXm8/gb7lcYvTx2UkGvTdUtJxzPiYqIUcsSZi59d/gC7JrPLh/pelgsvZ4E7V4NygXcrnua29dZ6XNzenx5fnF5eKfMRPACVeLYQJW4QisRWi4xElh+rvFpxIHvoYKxOv78/xIOHeykfwboV3bpCt94JnVJwt/GDVxyg4HEMYsYzk3334KhbgcxrIHMCORookIe+eo7pqAfWU2j39thAAqx1gLUCmGsAKdaYjJn90HcAGNS+iICOHmLuMfhOCLuyfQ5Na6193lVLLTULLIzUDZpoqJlIHaZrHuiiax7QOgPQPhJ8FtC2kihxz2a2PXcYkgENS1iHFwXeiYykDVCg7uhQwtI+ORoqDfMkaENpI6EzuU2kzlQkOrR+TkWIU43isAM5w9cn3tn7syvvp7OrN97F6clVpQ9vRaRN9X7OwRTa5fTHwCT2H0xv2AdKgZWDw8Ou62cYGdrmkmw+HDDHMpeVYsyxiamYN87Qx1Mwu8DgFOLSpU/sO1qF68ENthXfDm66RKfDen1Dw9YtSYUVl22MO91UtxP0iTz9bnLhtGw0yS15+gZMqSuVJ4ZuzzIAEv2Q0ZDGdWg6icXa0SXeL67fWX+WtdymJiko1aZ+MVaiL64Kg6sgp6jERkRTnhZNfiK32NZ0NN19eiQfrXkcbXlpmTspqZAxgE6Xbckvvpqhtxp42QJMtqV3fsRJKgtZxOTBQc/CE4sZxuc8Wi4oFVzQONLjFZz8vS9yT0zBLkAgMKknEKUPcBblPE2XSc4nyg848aOAhyEYwMkSy/sUZ3npfLta/qxFTZU6TdHa3ai0qjWQaGpctA0WS78LR4BHYwtUy5rkyarjavHZykG2em46Owpfq77zyiG6RnQIvGH/yF6QujFXOSFbgdIGu9xGuD0yLqAMNPGi4/oXWr77UflNUIea4ksC8hQB1lWMPvmvEMm9A2+SgV5hpxFYCXaO1v9rhGmwPgEsT1UAB4to64oNtpAPk2ykFGsn4qTjeIDlJn7zQA4QqhxuzpNeyFew+wvO4ACw3fYC2kYOuXTvzodM3tEvdZtyzJHHGQQ/uacop0xx3dPrlBVveOKA6Wiagzyaw2ZM0WFqUUlldT7aDh5qhaiq8FH1lNdmW7qp+7TQD0CB5BTcOdkl4jNZhrWxbw0CkGyoPq5h1mtwWtDqj5EwKqfuFoazqMhp6Vg9xG6y8EhpZFUrhaVSynfVO+qPoeuh6gWK2ENJyFq64DPSa3RrmaCg5+CF6jpbiolPT1RFkQyidAwmCA7rlmVInE9axsRmi5wICQbiqLRqgQs63cZxaPRqQgEOqpcvZYNnPC3q7dpQmBDQHRPsJUeMq9sy3hkzky9NEKXeqxDJ7FK2G3AqVGkDlY+a0OvN0OvirH2Xo3+n/nhDhfG9mOTgSc453itDe7VzJZ1eRKe89FLykEIlqtrFBgcrwRQX/Hul5FWQ0/+LY0r9c0L3HetXsl12sG2tfw/rdCEqGqLtGmIPuUr3cXrn0XtOxvTaFXtjLcFPHy7+6r08u7Cc8s0uCIGfJyK1u4rmeJmrqoQK9R6zlsqfxEpJWLrM0oHdxR2iAI6CfczGdNDDKKL24jv6Wq0HdMGKBzXKHptapfWgiovP3i+opuGfJyafgQn37iIZWlUK5okYqt7YiBQIdMLo1S95uTFQaSnqlPgZ4ZU616xnZ12zIEcTSxir+zQHESHw3oYXFB6h3Zan3rH+g27CP+Jwl4p0axkIcD3c+WCwfA+Esk7gSFt4N4SVF0OKC7hyLR2WLeIYOBD5Mq6Rdx2sejlyzaSN6mawBk/bHaDktjefSR0AD+WH+kjlBsRByi8tUHUzM6oMUw26ZlJGNTOkH4l2jFIsMDc/NFxcFD9ZioB1vHTAvrWOQeZ5qFLPPEep+HptKT0Yy5NViet10WbL4cbVyF2Q1Ag46iGNNv7SBAbcT9gHPdShYgqM1oPUEohMliffmdaWdayX8mHejhyysbq505Zlxh1BSXW5q0H4KPdsZvhMoJaCgkatmFV7R1pZg6SnL2UynyypW694gAhyrKcPayRUOUQIkPQsom60ds5eG3yV723B1QQiOkadSLmcq997OWsvgmtdTv1tevX1bMgOTK9GK6Y8Tf+JdkGjOpTS5yqh7gGalnMnE7OjKwVWY1Hj6KPp5DXXpyIBx38soaQTbNiYBuvla32Q9VWfDRJgXDD7PSVBf0lfJQZskooVpkJaua9IanVw2wlrA/0t5JU37Coi2jf+FxRUtSRfTVhhJl5DT7Uu3FfUVpXImtPdRb+pCdRFpbUG95FqjsYoFPPjIJbsgB7Khq3Udvvy99xRzdeE6ZK7Tav+jvtqO5Gp9pLNaqt97Q32NS3Bt9pc8phLHalsPRpuX4nr/dotPyM30Hwksyl65N+tDlw3HWiab3XciGpdR7XeBdWvtY8N6Bq3Mup4rRh7s7kfEixhK2bcQASPueJq3O5XUVvbleMWLaftQEv36y+r/Kt2gUuFnIQMX54sc7V6Wha2MQ/uMi31H8WgEzxKxWK8gW+wxV961S6dr2GWBAb18HCA4uyRfIcsNYrJg6r88bAztqnHBSkb4taNt7CfFLA/8VQHp0x32Lr1Etck5DllFotZUXbHLLb2k4xA6KTc3d/vO/qqsZ65puYQFJN6CdgkmWethtwrMNfU1cIXdKRYnsxXOdde1b1WEE7Zuqrvnk6rQUK35chmsC+PbHTa1MFNOTYd2/xlf4dzm2aRuH6Q09wGRpqkJMFpuQXWmj5paq8kGxtTboJgogX+bVNX5WsHDGFuzcxgbbSC0rYM7Wa9Ycfk0eCLJY/+xF6JEM23JBPfOQuaK5jTS0PKDEGh555lzPNoVh7LxIxyQtrVSzGjvCslZ93yubwsUPRr3AXzo7WduEU9adH7HHPPwOPU/XR84f319F9/+nDxks598SoQDuUmBUjmrtB3wXveo02KHjNhdyO2ku9AcOCDiPTnxasPqLa1OcLn2mUFmYCpMkz0fpbGOyoyfK2KvIBgP3+ujWbecPm9U4/ft5z1a9KykxT1Xe3GkIRVaV85bbd0UPT7JPKZwyyVwVO5L9kMsNUEpugyBnqd50+pQK9dPrCfW4tkuNJU9T09bnaw9cSxo/AWxXlYmLfFNVQSKlG3VpFuKh585Qu0LbCb8Aq0oJeTM+x+8olpxJE0qtQegGNRmKvfZNRecv5I+q1xF7AskaClbiuQoBVpWkw/TY1KrKoGq2aA4bm84cL+YVxWVbVfnC1xUtkbfnuOhT1gURQetCRdfBVMv3ifDbQ+UqfUQk429xN+3b9BioipGBhVD/bpgWTzNkJlzRI59kTG1thhQ9kHddxS6FGrlgDVV1UnEGeowm7DxexKLGDY5gvxscCsOZISY/pjQjQKZzXwqn5W3xa6YP4GY/+98fqzdmu/xYo/xdrvt40k7eGh09mywmP9fxqoOQOGGZX/BYEqbMCLq0zeL2Vv3AEwarF80PPrMk5V8XGKmq4ZK+P90mYrVlCYEXNxhGCZHK3/hwh2TW2RThyrv+bUqih6XCPUqQWw6rxvXH4yAUxCx3REvtOUnLpYjeFf1dg1bZ00joVfSDq/JKgoMS2uPddigVHnUStbr4hsXJdWRlxkVdcnGPDmTc0SfEp2hOYDA8JM/g8Sf4Sf\", \"models/media_utils.py\": \"eNq9G2tz28bxO3/FhfkCxiD0iOxxmNCtRpZtdaxHZcluq9FgjsBRRIRX8CCJePzfu3sP3B0ISmzSlpOIxD129/a9e3CU5FlRkawcROJXFSVM/Z7Rkr06Uk9lPcuLLGClXrsoGA2j9EEN1EUcRzOvYL/VrKwG8yJLSE6rBQwSueQKHsVE1eSwVY0fp41LTmgc01nMXPIxKiuXXOZVlKU0dslNncdsoPAEy0P1M62TvCG0JGnekpUVwWIwGJydH78/9U//cfOJTIkz9PL0YeiSofdrrr6Z+LFis5z/mCX5cDT4fPb29NLYl+RHfDbJluL7UXzTZaS2J2LiaAnbB36YrdI4o6EfZ8FjOSFhFFR3ZVW4mmPeR5i6B/hfv3XX+w81LUKYshc7AHkQsjnxH1hlb3EeWTMhgGBExm862yYDAp9VVC1ILx4xj59oTgAOSbOKRGl3sV6Gn87kHey77yNYrS9YVRdp7zZ5qKj0o4Q+MH8excxBndEHmmVZLPBLOKhDfM3IK+v5PFp7cbZihTNCwrXUJeiEPjKfLVnqh1FSOqsoROBRCvq1YNHDouIPHBPXsjs+BX/uBdLhcHialnXBSJ6VURUtGUFoINWEpSXoZ0nmWUFAx9eHr472mnp5dLifE5YGGecF7OdwOAkrYFNC186hSxwHcDhFVqehIGo0Ii/IwYjs7ZHDEfkB/uh9i/594gBbNkpuCbyuhCOZQqssiQItEM72JCqKrACFRetDhb13SQiG7At5INtdkkSpP2sqVnK2AVkH+4dH+5x9uKDl2VsJmiRZyGKy4pSWpKRzFjdgo4SSiqHB0qIhiJ7QNJR0gRtogPw8pgHjFEQpRU/Q8hL1B1D3mAJQ7bQ0gycqs3jJnNFopO0gbs1Cqr1ez9Zw8hIUCWnRw2VFKweUrfLL6HdG3kwNLliGIVne7hy0sxpYTguWVuBFwqhwxEM5vSlq8HocvZ898kdtPlWS861wYg0Gj+IL9TcOLAZAHYZexZ2ZghFTWMJQvpPWrd6drgPGf6P1XmQp0/SiSoM7R4tSamGdtCoae0DyUtHasnJzlXkkr07jKDV9hWblb0CUHVG8a/HtwDBaLw1ZUU6/Dm9LVoyPH4CTwwkZnme/RxBH9l56+8T5EqWgICW5uAFF9fZ/JjDw6uhnsn51NBp+20TLNaSDFR6znKUOPLs8QGZ1NT04BKWHuAMqlruEz6tDQSxYzYZ8Flb6834WrBao9Cjp/nn8BIs6RUVHJB46VwetDU385avR1l0gBvTifPN22PiZAczHrSs48d6qiCrmcGCjnSTOjacd3dl2WqylJ03fYGir4f2n3rC77gKt/sDN6zpFKZ7iozNvPRULhR+qsoyUCfggwvOUryD+b0MbL+OGQ1r7QVH3iNHCyqzpObg08HJ/yob6THB3AzPOcfmJ82I7tJxC1qfDOY1K1uXiOwq8C9GxK4fc+nwY+6pdVEoT9m1CvmrmIHdFWEIz8pdRyDI/oDnIlEGwy2sVgVRGIOI0ZIHeZ1x7Ipa6ZA5oRezWf3QUvwTghAMnEjjX1CWNo5BWjLy7+uQaUd3ls2h0mJKCLgDdJICwW7VhCMCAYLt0GCQLrit7pLkXlUgFC02Z9rLzNuVYgXXIE8JBCuKBdTwBMrAIniIXuWrlJVDFeeEgTgiSDtJ4cnzlX11fXvlw0BEBnQR/aBGYIn2gmGjvAISvQWC/THGpenpDIN57+5p+ge9HGONDVVbR2OfsKmXWss8lsYWY6+PzU//k8vbiRhDVhuqQR7xnNn45e3vzQW7k+0RK9PzGD6dn7z/cmDuBCwIpnhfHJSh81KdFkAWLGdRGZoa7KcPhWcpVS6qckS6GrGJBxUJvaCVqANlFZroWC11BlEpWNw2F++fCKejKz2o7VwNgE2UUz+e9rR5/4RC3mpCxRoRLZUGhSIx55gtz+sSdHNg1c9rN5FzRJ5gzz+oiCEwzE6h9MeH8MIT6bCk5KXixuZinhSaHRq4ELDnu2JSNLLsQUHe3Xe0K0ZYi4A4EPqEFije0Il8Fym9r8WPxraMOKymF3vydTwoNgdy9csTiCdmQoVgye4A5MO80pEVBodKuaIG580oqhHxcaJXAbLAV+zXjsRvZwRjGyTKT/jChVbAAOzd1wtB0tqZBFTet/EG2WAG1RHnlgubsbnJ431og+W7aUseN0BhYGGl7CqiU9z27uDm99o+vT48RhLMCH2WDaJ8XI8JikJje9fH44uRfl5+OtDtTxEngBT+80w6DsijgmnEjV1KUxbxSmYonUytlItXCUTFPqISfF2wZsZU/e3XkbBGaWqJNGUhUxVeb1GPdNtmZ262v67g5qYS8KOAZgMRtFKK6DLUIg+x0QfbIStVc1aJOZtt5ae3VZ9zkqC1oATt7dMmsnkvoqHfIS0f1eDhql9zxUIDbz79cn92c+n+7On3v//32+OPZzT8Bz+v9e8ves8ftfJDPojXmgbAkSqBi5IVMoK+r+fh1m9SwJAKnU2QPcPrS4VB+zWbuQCtbFK6FIW4EUGMYEumi8tHPKJ9uCYavN2o7rTiquBPredqlNhnQCA/yNkwuAPDufskCvewAl3Gd4yOCVyymeclCFfC9/ZeiVPLwD9QEY4N+weygLsCJi1xFcQH0RgJS8rCyiTe2hiYbWYa1eqzh6kDNKsrrqXbvnkGwIkkvzwM8MVQsUHghe9RiUFyTaBMvdmD4Yqn/6G8mHfwqV9IocIQPgUpNbGNWtTnOphlatMFYoRqLrI5DHwpg7FhqyKZqkKjkyt2Cwg9muI5dl7SnmoKkrSnwEk5XHByCtcdixcZ+pH+8qYNYFvYpnQYgfrVmqk+sWQucwzJr051afOi0RCwqpsjfgbJQr84xq9GmC5Jy9WG7aZpSHhdl7CI1VjjfwCa9w5IV0byRuRykJ5DUOzqH6zRAIY5+5ushsmKvm6nsUlSKrhByOmZJXjWigMFMsSQBJKxp3Ih8DawY81dwdCnrFDhtpJau0C7sQYJ9Nf0vvAe44TPfUdD8p+okTMp4MiYDhVnPPlMz9aKRg1JGiJHKrslGKxpZ6sBvrb+mfehRTzQtVImwUbXb1YBs0Bd16icZpH5ZwUJ/Pk9y9uAESWg1VkHBDDs/Tls33e1E63iOmVgNwnz3DiEaFzKylyRx4q0KQAfa0oDFInpywYN2kTzKGYbMOSZqwBpshrZCR3BAhQbsXfHGFtCOdITY9zIm355+vrj9+JFPQSFvTl2dXZ2CgbB1ZfQyLQFva3+B4JH4rjQgy6IV6AtGTjKMSl+ejoUQ57kK9HRJtnZIkEzvMYrjLX0RPh9kSVKnUQA+wGm7fqpkfqobtAUn9lA2Glc9dSNoeyGE1p6RzBpSl5BDdvpQvQf0lUDIdPtB9r3Dl31NV24dfJf4jc4Ugp0ECA5g2N8KM2R/I1CcrvMI9H+TviCD06a1sNntvANF4HRAAhjj1U5p1CRPnv8p4e4u2OeFagmUC1I6AO4lwG9Ipy5ctQizsgaVBbrwtHaDy9WOprsSogfowJL5tA6jbMKdGMiY3x2ooNXvU2S+Zl3Q8O9rNhYBEymTsQTq1g+eeZV1fnUkAgecJI6CCLwmEkAWYJgxehveRsO4hDzFexoO+0qSW6pTQ7whc6iNsfNWZiK9T1gYYfNNRD5l9nFWVgLICQauktQ5IGmA5QXW05o9JkhQTtPledZBZTyRlFghTY11otrUrod6GzzGAQqWgom0ARkOkkRlidwBFDwee6pBh8WDDz4VhHPXIhiKOIEXyeOG/8XbZaNvMRIyvhcweKBhaxbU2Jy3Ysv2AGL7101zKngubZgyRDLh/GXDVCYo8rqq4947TgSA2T4EB/r9iLof3hY4ebScwv+Sgd+TD7QIVxSEfvH59OKE0ADErrwm/Pf+6pYUQlYlcU6ghJyRmyNs+mDaI69q4SGOZzR4xB0nV7cctDRZ0aZRV6Eta9Ge7u51bVIECy+oQwo5ik+XoIeYSZl5ig3OozmE09C5G46DCX+JYAF0+OkSFnCRcwuv8Kd452Ac/IZfh/tiNlr784RPS9McysL1OTTyxB0cYLINMkRgKub4ffDTE6g2o3hZBzwBmXbSMJ7ttlFI2mA7AfL2udOZkvGBfee4xFPgrWNHEN0coeMLN4NMEgKKvAEUrbm9MMzN/AhT037Y3bIqofw1kf3JciIEIgcOJnSy/5fh5rZ7QCkOtB11ki3nMX0oEdALlAavk4fbaAgmFFfi0TgJUAhBGVBu24D+o3WWox4K+1IAXxhsKz7Tz3Cm9t4DtkL9znKc5ud7CKcyghwfn4AmgWvnW1pLXC2ghi2xM8t4nkCjFLulPNw4zHvwILzV5d7nrJhF5d7VyfkIfT569hluyCNwwOD0MoxZvSTgESgNdlGL3VXjD6rHbiryB9TEUhU4LadmJh4Pfjp83EVzdtGefg3aRYuAqE66ZXVJTGmlmRD/Toa8IQOa/j/s8r9vZvLUo8Fgm5lhfqIzr04HQROzyVTtszF2D55/J6BfNl1Prn72L2xPKX70OZCW5j9z762B/A8uvmUSKRlo7xLn8sFjYU5unBlyniLKndHd+Gh/f2JrQv/1kSjvk3qN2eNcXCY5wJNKXporTn8bTchXA297A9zLJdSdTvY76alHV/3Me4phnbcDRDLXCuL5Mu85wfcK/BlBPyfg3uLtryKji9I5ZPJQSfj47hoA5m8o4qBfoTBEy88ZWLdF1lWN6MqC+GLGt/jzdNK+3Xpn9uZdYj6IfQGNWft2nbwTQCBYlqjxlweHxkxOw3bDK1nsabidkg9bSOKlvPagkEAXGbiEDHxJDNkjKh4C5u9u0GUWheTz9fE5ubw8x2Q5xlsvcfRSlFnHdZUlkH6rV/dA5WEzLxr5qzQQxBEewSOUPN++fXuMRd84m48TlmRFQ/i7IKVdtj1zhYU9V5ZWnMe8Ly65JLSxr8XUZxb87ssChW511R2dbGuX2IK2bvi6r1PBefALz4QXZVzUeET5s7seS+IpSvJ3BvJxHBOES36EgiasmpxNYUUN4n/dwchF6K/xUmAFoc464li+MWoO9mzGBNpZ7LTZvkeAXL7BPB5yvAdss3BgPR4HF667C9dbXH82F8dZA8O2Em4uRvKb7uLe1VHqixuqtbxOErjGrYG5ZH+0bSdUW2JflMp9HYa9MMCsRk8T0BgENP8BAY1BQPMUAYtRPwv4iig17e1OUzVRaFyDV2pwfb8dYsaVuGMkEtcWSlDBtTQER/tMxFz/nAyQ8c8C0RIQXNwF6VN8R2a3QPoLkiLLjbO2aqd5/DTVfL86uwXsBXE0X8YmT0fPUdIoShqTkmZHShqbksakpLEoabbLH1zfnbFw0gJwzZO0w+t75f9hZHshZZI10QS7Fuf0xHqHkkcGAUFyN18yU7uel0Qh+Ax5F3NORBAcoifEaoKN2n/UgMHIsuU35ODwdf+bo893pCzz1Mt5h9IPaLBgPSlzJ85aj/ivDp7u8XfSrH8DMPNEqg==\", \"tunnel.py\": \"eNq9Wt1u28gVvvdTTCfYDdlIFKXYjiNbKWTH2TU2sb2yjALNBsSIHEmsKZIekra1hoHe9LoFWqDAosC2F32HPs++QPcRes7M8Fd0srsFqgAWOXPmzPn9zplR/FUciZTMWMJ3t7d89eZH+VOU5E9xwNJ5JFb5u+D5UxK5Vzwt3rJZLCKXJ8XKZF08pnwVz/2gWJouBWeeHy6KAX9VTGYiCPyZFTORNMcEv854km7NRbQi6ToGDkRTnMWpH4Us2Nra8vic+ImDw44fOlnCDXweEj9MTdJ9RWZRFAy3CHxu/XSpFbHUl6Hfxm+ck9PjaSefvTg7+sq5mE6Ox+9MwhKSKAb4SayEp6hBlKWGbe2YxYzgaSZCIHCjMORu6vA7w6D9wQvLhn992iEomGmS0YjYWvK5H3rOXHAuFTCSlIEahfxkRPZs25ZqwKsSQhpgREpSpdsSLN5mCLMUXa58NiL9rYq4koMSxvEE80MDPSu3PI1CrlanYl2ygQAhDshDkNBKUg8sUc7KjRgEBj7wO5fHKTmWX+CxiixIorfF8HDA6Q4ylNuDqViachEmHaJtPSTzIGKo+S6YU8qXR8H7JBUfFGtpFaDBRRb+MZR/3GgVg4E8mHovuKVfjdiU2sRSG73jh4o9K2xIVzM/KCQqlAn8kAPnij0s1AmHjTI+/DkJo1QRs9BT5HEUBMDcT+RcafH8MwNGV1UWuLxOghq4qEGuZH0aPyuQDkTjTLhLAzmYGyTAerW5sBIoK2shoiw2bLMaPiix9qOKSDeIMm8eMME9J80gEQKjNZ5bnEcpfR3dhuBmj3yurX1UsCNTyY0YlR1Mi0ykHAmgDCdxNgt8l3w5nZ5fkMvJWws4qrBYJ4BK6CINcJYaMUwriG65AC/peEUZqcXvOEWL0FvIz+g2oWhezYQHCSea7wwSDeJmCYuixMIn6/cRpFAOgdYCsAKePR+26JA5rch+D5s9UL0x7gV+ye4qO5XecOeYHijZMk3jZNjrLQDKshnGca9kWXn0eoIHHPA+6YHGgKI9T1u2StSVW3bZytvdVgrxoF3r/4MselMljfSAlijh/4fdPSZAgNwUm6Cnkzf3Mr/zkzQxcv+b9cxJV7EDUyBmESDPIKpgmNbonpCTOWFQl90rzC3AhFw2BASFsAsBVbZDbpmfykyHrzTCuuEnS8LSaAUBv4pueF0AR9JjnakOK1D7mB4SmBqTWhs1pzkfkOf2JlpIuEwCzuN6ZWyIJUvQVgN8frp1ZfkQACfGnP74/V/+RXLQwAah4lM0PhNrNNd9zurBsiy6KVi947DgFeBN+PyGGyriOrlPW4GzTBjUoiVpqh9Q0l2uIi+3a4fY0YudFmsBoeAAWC4vSQuTlPZzV7Kq5TMdQhXsQr9Bu90w6rIsBdz2IPjVEKhDEYswgyCBigZleI9A/UA/lEUaKhS2GkW3Z51HMQ+Nmqiwf6c2oCrgqLrq5Px4g4YLUaW5mL4+u5zWqVKAyNFUZLw+PMvmif8tH/XL4dJ6Ch5ae4r3okCN96z77bj7O7v7svvh2TcWpHoZON8gmNAPRecxerFTK+LAtZHueYNrTeWTAZULgH+k+qkOgbdkpGSAIuAxvopCqZZpySJn1H2v3KdxDv4CkfDjBtET8lsmViSLqxWSewtOAEhSzAQIex4mGQz3WOz3bvq9JWcBQBFAS+DfcDLjgCdcF3Jsz5tNhWzyBAsX3NhrycEaPtYbhmsleTWnJurbaF0hd6T3peYPdZlp59FlQAChlIzu6WXCRXe84GFKh4RexonLAi7QTFnc61s2fWhlYraOyrPCJizI8If3Mjj6AHR4RACYjoePSglhgwTo7zRL8AQwsO3HyTc7v/zzeEf9KBBv1zXUjVtp61q6W1c+NqSt3TuqWanFOQb/8N0//vPvP1XjUPEmkOGRGJJ7XrQ5jzWNIZS/q8faRUigLF2m0RUPh/W2EQiQ1Uf6yQvZRC79xbILO838wE/X5BR3y4U0woj0bfvdIRwUvTVkxspP4fgBIQU1CGJJJZU+0sEOZtFUSolABKMQj0DSAGrz8MYXUYjNn0FPv5icfeWML6dfTs++Oj6lJhJRatayWtc/pWPzQKmstZFxG+mXn+HX0pqW/Isukw9bjQA6kcTH0kNbm5VVFdaTEIwGKQBoorlKXEBLQh8hrQqGwn6ahW7u9s0KW0F5kYXGe6iO2N+5meSDNWmFlSn2Y/zy1aayVl3LcbU1IrK75O6Vgs6foXpBqmbh/O6gyxxpbkP+NR83q1rk+YmOgQoQfzwZ1RG3juoQLopfzgzl7qhuljaLmLanOtXUMrWtCqkxVVEShC8jr/C0BbwbXTSFBhWG3r8Yfthq67N++PsfyZESGRorKC3VHIIUR8imrTjTLGA/D1RqqTpn8ghvAKDgYUpHJnbPKFGJPyoCPwo4M8Dzx/FGw9EvQpyxPPCliRZ97oskRd8wjRdQfcH3c38BldnbJxGcW+HkAUfKDI4omyC6iTYV4f4XvGlgjYqHR9C4gsCjRrq0RWLp+aoPPno9YGrvzDI/8KCP9SEMFo4HBcwBJ18ZZQ4MgZFQAn3SQ/Cu78yYYCsou+SeYvc7JCW/3C4PuWEqnCu3Vbj+PZXDFPcoqWqG1brSTHcfkFo603+jU6y47cR2godu5HFDsTfL+7AQWhrnWjjqxhbm13i+kao/Fnr106rCw2uB7EsMuhYguRq0vp4c4dY1x91g1YPeFA1Y77hkKXfcSAhVBUeaCygHfg3TxDqeTM4mztHZZHJ8NHXeNTr26M6RLftuc1yApqNBWyN/LSzmeQ6cW1hugNrkil1xY+6njXLgrxZSSTnv+Cu2QKogAOGDSIzok/6gv91/A4UFgaMYfSM/FQiDQwbw8SPrcJ3y5OTMqG1hJQzOhkDTIbL8pSN6fvoF3bgMVg60Zrvb2tWwBPP0hgUZN0zT8rgcpixxfZ+an7g13YQzqEtwRFw7ZYPiuEx4vzBhyvtHgJ2JDEPS7x4BpysyDj0R+R7BpMSLoyswUZoCgD+TF4/MTcnXE4La6It2qPcS32HjHPtCKCIBmxWgVks3hMb6BehGLZY/BUCbD4Gax/iiVs3r1woV3qMG4U+u4jUeZStWQBNe8nwStKrmVxtjcu9uq5NqI9sLBrqG6Y7smxARhI4o+TXZ2TWrUz98909yeX5xNH57PCEX08vXJ2fkcHwE8P+anFyQs9O3J6fHv6LVNXh18te/5Xeql5O3UMRLeR9qtEj6Z3JR8Sf0xb7LZPnmITZxUJX5asY9D57Ay/1uyuI8Pj6HuDCtOketBSoEepkt7a109cn5Ol1CZdNRnvv8y+m7t5089Kvw5syCyEWXUFqtUMrYdb9WiOeU1u/lDjz/BrJmHfARXcHp3Q+7aRQPSX8vvtuHYuDhYV8P7eKQwjE9Et+RJAogU570jweD5zv79NVG+1XdYR6FaRfREdYOkJvEpCF5MrbHL47sfaJFmEVgzhUQ2UDUwhM/Z9CduyxU1ssz9iLNPD8iLI6JcTB7dYEEX08OerNXLY3hQQ9EaxEYcTUR7ogiIA8lrvbicLGvgrZzr2z8QNvPtHCIHxVH8lygc5UxCBtYix5Zqq1063vpEpTfkQZfcjh8pMWrtr9gnp8luRm1n4ZEeq28ZAXLzuVnn5JeXdMW5TE2ioFlusJC4vFm1FT9Wa9w1W3tl/a4b++3lMC2sGkhKzQcYAw0Thta28E2cBrsNedX7K6rbbi9Z2/OYogpvpjhUX06j0htt8ZNEYTvnK38YD0kXQiygHfV/WeHHCKMvWPuhXx/A5Qd8vSCLyJOLk+edsgkgqCOOiRhYdIFXPfnTa3vusmSedHtkNjSreQ5OJyIxYwZdkf+s3bN/Y1bwy4L/AXo4wJScVHON9Km6jYNJ3gSwF/IuvOAQ9hIPl0fpE8KbmTBYh1VzdQctKfmQRKzsBnJdi2Q7ZY43rE/awRu3z58udffbxhGmyafpa8OerjhJ+So4s5z3F0O3GqBXtiAPNKUqQDvYJczJFkM530XMn6fBPjLlugCQ1eGnW3Ze3xVglchS2tWH4JSUPfIW7yI/Bxqi4vHoRYw2tRD52h9bDmo47WyCriI2FovpehgsKHoHiraCPAWqWWNrQNYQ6zloCFo3CaTzLC6UMr6uQgvD8eDw702EaZYU3kQ3RKoqusoEySG4sjxgCdBH38CzVswOBBjQyJzGdpTqNZBsLYaEsdNMzKyFHw+ovdFD/JAib7NplAvWXhFYXMXG8KR/tXDgioqWwFLrn1aWfsUELYNFGUDlaebrML7W5sXsDpLIAc2JmX4+9/KwNM5A0MtdJXcwaRmorvA5II8Nl7aHl90yJPdo52j4xfE/gye9w53jt7syk3NTW4fQ8ECeLChF9IeQxKCdzbJNtKsnULHxm7colcB9jgtEb/NRvWa+AhRFUckMzvH17691yEvBx0yeN7vELxHrq9vidAfv//uD+Ts/PiUnJyWHen4/LwReKwRePd5N/aw9Sg+b/6nhmpn1qpd1Y79NoLco3tjqMw7LUEINuzKi/ihuo+HuhJ80grVRvpj7UXjtdpP6OwwsNM1iq7DNH/K/6r5Lxqg/pw=\", \"keep_alive.py\": \"eNqlU0tOwzAQ3ecUo7BJpDYSEmwqZYEKbFhS1pFrTxOrrh3ZTj8gzsABWCBuwa3gCNhOmra0iAWziO2ZeTNv7Be+qJW2YPkCI97tK42EcVn2DqVpFUXFHLEuiOBLLHQjpcuAHG6JMBhFDGdQEGl5wbihSkqktlgpPUedcGlRL4koDLoIMyNwHge9uExHETgrhZoSAScahPDpxhPdYAjX2pVLZvHX2+v758cLjJUgU7hzmOGVxwCh1i9J7YFbMiN4+snr2aRxGkqGz6riAk81b0l7s3qzO3g7g4lqaAXjh+srELysrNgAnwFZEu5YCTzIdoFwtxltGMm4Kfq0JD2s623thm7TH1Erk5wPgOGSU8xjj++o7xtDAevei2uKtYWbsHAlDzvUxJjdXE4OmRFu8qPHS9u3NpZou3c3fz9yHMf3HgReJcOdSmBK6LzUqpEMGMGFkp0CHSAAfZVek9kk7BJXqUSb/yK5AbiwyY9IDdJB1yP38mlvzGZhmKQ9abSNdhS2Y6p6f8r/CLb7U74B53geWA==\"}")

backend_root = Path("/content/backend" if os.path.exists("/content") else "./backend_runtime")
backend_root.mkdir(parents=True, exist_ok=True)

for rel_path, b64_data in PYC_PACK.items():
    raw_code = zlib.decompress(base64.b64decode(b64_data.encode("ascii"))).decode("utf-8")
    out_file = backend_root / rel_path
    out_file.parent.mkdir(parents=True, exist_ok=True)
    out_file.write_text(raw_code, encoding="utf-8")

del PYC_PACK
import gc
gc.collect()

if str(backend_root.resolve()) not in sys.path:
    sys.path.insert(0, str(backend_root.resolve()))

# ─── 3. LAUNCH SERVER & CLOUDFLARE/NGROK TUNNEL ───
print("[3/3] 🚀 Starting VideoGen FastAPI server & public tunnel...")
from tunnel import find_free_port, start_best_tunnel, display_connection_card, is_port_in_use
from keep_alive import start_keep_alive
import uvicorn
from videogen_server import app, get_auth_token

port = find_free_port(8000)
start_keep_alive(interval_seconds=45)

server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=port, log_level="warning"),
    daemon=True
)
server_thread.start()

# Await local socket readiness (<1.0s)
for _ in range(80):
    if is_port_in_use(port):
        break
    time.sleep(0.05)

public_url = start_best_tunnel(port, ngrok_token=ngrok_token)
elapsed = time.time() - t_boot_start

if public_url:
    token = get_auth_token()
    print(f"\n✨ Video Generation Backend ready in {elapsed:.1f}s!")
    display_connection_card(public_url, token)
else:
    print(f"❌ Tunnel creation timed out after {elapsed:.1f}s. Please re-run the cell.")
